In [0]:
numeric = spark.read.csv("/Volumes/workspace/default/bosch/train_numeric_sample.csv", header=True, inferSchema=True)
categorical = spark.read.csv("/Volumes/workspace/default/bosch/train_categorical_sample.csv", header=True, inferSchema=True)
date = spark.read.csv("/Volumes/workspace/default/bosch/train_date_sample.csv", header=True, inferSchema=True)

print("Loaded successfully")

In [0]:
print("Numeric:", numeric.count(), "rows,", len(numeric.columns), "columns")
print("Categorical:", categorical.count(), "rows,", len(categorical.columns), "columns")
print("Date:", date.count(), "rows,", len(date.columns), "columns")

In [0]:
numeric.groupBy("Response").count().show()

In [0]:
from pyspark.sql.functions import col, mean as spark_mean

# Fast 1-pass execution on Spark cluster
null_counts = numeric.select([
    spark_mean(col(c).isNull().cast("int")).alias(c)
    for c in numeric.columns
])

# Driverside conversion & sorting
null_pct_row = null_counts.collect()[0].asDict()
sorted_nulls = sorted(null_pct_row.items(), key=lambda x: x[1], reverse=True)

print("Top 10 columns with highest null %:")
for colname, pct in sorted_nulls[:10]:
    print(f"{colname}: {pct:.2%}")

print("\nColumns with >90% nulls:", sum(1 for _, pct in sorted_nulls if pct > 0.9))
print("Total columns:", len(sorted_nulls))

In [0]:
joined = numeric.join(categorical, "Id", "left").join(date, "Id", "left")
print("Joined row count:", joined.count())
print("Numeric row count:", numeric.count())

In [0]:
print(date.columns[:20])
print("Total date columns:", len(date.columns))

In [0]:
import re

# Extract unique station identifiers like "L0_S0", "L0_S1" from column names
station_pattern = re.compile(r"(L\d+_S\d+)_")
stations = sorted(set(
    station_pattern.match(c).group(1)
    for c in date.columns
    if c != "Id" and station_pattern.match(c)
))

print("Total unique stations:", len(stations))
print(stations[:15])

In [0]:
from pyspark.sql.functions import col, when, greatest

visited_cols = []
for station in stations:
    station_date_cols = [c for c in date.columns if c.startswith(station + "_")]
    flags = [when(col(c).isNotNull(), 1).otherwise(0) for c in station_date_cols]

    if len(flags) == 1:
        visited_expr = flags[0]
    else:
        visited_expr = greatest(*flags)

    date = date.withColumn(f"visited_{station}", visited_expr)
    visited_cols.append(f"visited_{station}")

print("Added flags:", len(visited_cols))
date.select(["Id"] + visited_cols[:5]).show(5)

In [0]:
from functools import reduce
from pyspark.sql.functions import col

date = date.withColumn(
    "total_stations_visited",
    reduce(lambda a, b: a + b, [col(f"visited_{s}") for s in stations])
)

date.select("Id", "total_stations_visited").show(10)

In [0]:
from pyspark.sql.functions import least, col

station_time_cols = []
for station in stations:
    station_date_cols = [c for c in date.columns if c.startswith(station + "_") and c.split("_")[-1].startswith("D")]
    casted_cols = [col(c).cast("double") for c in station_date_cols]

    if len(casted_cols) == 1:
        time_expr = casted_cols[0]
    else:
        time_expr = least(*casted_cols)

    date = date.withColumn(f"time_{station}", time_expr)
    station_time_cols.append(f"time_{station}")

date.select(["Id"] + station_time_cols[:5]).show(5)

In [0]:
from pyspark.sql.functions import col

# Total time span: last station timestamp minus first station timestamp
time_cols = [col(f"time_{s}") for s in stations]

date = date.withColumn("first_timestamp", least(*time_cols))
date = date.withColumn("last_timestamp", greatest(*time_cols))
date = date.withColumn("total_process_time", col("last_timestamp") - col("first_timestamp"))

date.select("Id", "first_timestamp", "last_timestamp", "total_process_time").show(10)

In [0]:
from functools import reduce
from operator import add
from pyspark.sql.functions import col

# Filter out non-measurement columns
feature_cols = [c for c in numeric.columns if c not in ("Id", "Response")]

# Build and sum null indicator expressions
null_indicators = [col(c).isNull().cast("int") for c in feature_cols]
missing_count_expr = reduce(add, null_indicators)

numeric = numeric.withColumn("missing_measurement_count", missing_count_expr)
numeric.select("Id", "Response", "missing_measurement_count").show(10)

In [0]:
numeric.groupBy("Response").avg("missing_measurement_count").show()

In [0]:
lines = sorted(set(s.split("_")[0] for s in stations))
print("Distinct lines:", lines)
print("Stations per line:")
for line in lines:
    count = sum(1 for s in stations if s.startswith(line + "_"))
    print(f"  {line}: {count} stations")

In [0]:
from functools import reduce
from operator import add
from pyspark.sql import functions as F

new_cols = {}

for line in lines:
    line_stations = [s for s in stations if s.startswith(line + "_")]
    total_in_line = len(line_stations)
    
    if total_in_line == 0:
        continue
        
    # Coalesce nulls to 0 to safeguard the addition
    visited_flags = [F.coalesce(F.col(f"visited_{s}"), F.lit(0)) for s in line_stations]
    
    # Calculate missing percentage per line
    new_cols[f"missing_pct_{line}"] = 1 - (reduce(add, visited_flags) / total_in_line)

# Apply all column additions in a single transformation pass
date = date.withColumns(new_cols)

date.select(["Id"] + list(new_cols.keys())).show(10)

In [0]:
import pyspark.sql.functions as F

# Build average expressions for all lines at once
avg_exprs = [F.avg(f"missing_pct_{l}").alias(f"avg_{l}") for l in lines]

check = date.select(["Id"] + [f"missing_pct_{l}" for l in lines]).join(
    numeric.select("Id", "Response"), "Id"
)

# Single Spark transformation pass
check.groupBy("Response").agg(*avg_exprs).show()

In [0]:
check2 = date.select("Id", "total_process_time").join(
    numeric.select("Id", "Response"), "Id"
)

check2.groupBy("Response").agg(
    F.avg("total_process_time").alias("avg_process_time"),
    F.max("total_process_time").alias("max_process_time")
).show()

In [0]:
date.select("total_process_time").summary("min", "25%", "50%", "75%", "95%", "99%", "max").show()

In [0]:
date = date.withColumn("log_total_process_time", F.log1p("total_process_time"))

date = date.withColumn(
    "process_velocity",
    F.col("total_process_time") / F.coalesce(F.col("total_stations_visited"), F.lit(1))
)

date = date.withColumn(
    "is_high_dwell",
    (F.col("total_process_time") > 43.24).cast("int")
)

date.select("Id", "total_process_time", "log_total_process_time", "process_velocity", "is_high_dwell").show(10)

In [0]:
print("Numeric columns:", len(numeric.columns))
print("Categorical columns:", len(categorical.columns))
print("Date columns:", len(date.columns))

## Phase 4: Feature Selection

### 4.1 Consolidate engineered features into modeling table

In [0]:
engineered_cols = [
    "Id",
    "total_stations_visited",
    "total_process_time",
    "log_total_process_time",
    "process_velocity",
    "is_high_dwell"
] + [f"missing_pct_{l}" for l in lines] + [f"visited_{s}" for s in stations]

model_features = date.select(engineered_cols).join(
    numeric.select("Id", "Response"), "Id"
)

print("Engineered feature table shape:", model_features.count(), len(model_features.columns))
model_features.show(5)

### 4.2 Rank raw numeric sensor columns by importance (shallow tree filter)

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier

raw_feature_cols = [c for c in numeric.columns if c not in ("Id", "Response", "missing_measurement_count")]

numeric_filled = numeric.fillna(0, subset=raw_feature_cols)

assembler = VectorAssembler(inputCols=raw_feature_cols, outputCol="features")
assembled = assembler.transform(numeric_filled)

dt = DecisionTreeClassifier(labelCol="Response", featuresCol="features", maxDepth=5)
dt_model = dt.fit(assembled)

importances = list(zip(raw_feature_cols, dt_model.featureImportances.toArray()))
importances_sorted = sorted(importances, key=lambda x: x[1], reverse=True)

print("Top 20 important raw numeric features:")
for name, score in importances_sorted[:20]:
    if score > 0:
        print(f"{name}: {score:.4f}")

###4.2 Rank raw categorical sensor columns by importance (Chi Square)

In [0]:
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import ChiSquareTest

# 1. Join once
cat_data = categorical_filled.join(numeric.select("Id", "Response"), "Id")

# 2. Hash string columns natively into numeric indices (bypasses StringIndexer Pipeline)
hash_exprs = {
    f"{c}_idx": F.coalesce(F.abs(F.hash(F.col(c))), F.lit(0)).cast("double")
    for c in cat_feature_cols
}
indexed_df = cat_data.withColumns(hash_exprs)

# 3. Assemble and run Chi-Square test globally
idx_cols = [f"{c}_idx" for c in cat_feature_cols]
assembler = VectorAssembler(inputCols=idx_cols, outputCol="features", handleInvalid="keep")
assembled_df = assembler.transform(indexed_df)

# 4. Run Chi-Square statistical test
r = ChiSquareTest.test(assembled_df, "features", "Response").head()

# 5. Extract and rank results
chisq_scores = list(zip(cat_feature_cols, r.statistics))
ranked_cat_features = sorted(chisq_scores, key=lambda x: x[1], reverse=True)

print("Top 20 Important Categorical Features (by Chi-Square Statistic):")
for name, score in ranked_cat_features[:20]:
    print(f"{name}: {score:.2f}")

In [0]:
top_numeric = [name for name, score in importances_sorted[:30]]
top_categorical = [name for name, score in ranked_cat_features[:30]]

final_features = engineered_cols + top_numeric + top_categorical
final_features = [c for c in final_features if c not in ("Id",)]  # keep Id separately

print("Total final feature count:", len(final_features))
print(final_features[:10])

In [0]:
# Clean feature lists to avoid duplicate column collisions on join
clean_top_numeric = [c for c in top_numeric if c not in ("Id", "Response")]
clean_top_cat = [c for c in top_categorical if c not in ("Id", "Response")]
clean_eng_cols = [c for c in engineered_cols if c not in ("Id", "Response")]

# Build final table
final_table = model_features.select(["Id", "Response"] + clean_eng_cols)

final_table = (
    final_table
    .join(numeric.select(["Id"] + clean_top_numeric), "Id", "inner")
    .join(categorical_filled.select(["Id"] + clean_top_cat), "Id", "inner")
)

print("Final table shape:", final_table.count(), len(final_table.columns))
final_table.show(5)